# 01 — Activate, overlay, and cutover

This notebook is the **config story** of the local mailroom sandbox. The
sandbox does **not** fork the 13-node LangGraph pipeline. It rewrites every
taxonomy agent onto a local serving tag, writes
[`data/runtime/taxonomy.yaml`](../data/runtime/taxonomy.yaml), and
monkeypatches mailroom's hardcoded `CONFIG_PATH`.

Canonical code:

- [`src/mailroom_sandbox/runtime.py`](../src/mailroom_sandbox/runtime.py) — `activate()`
- [`src/mailroom_sandbox/overlay.py`](../src/mailroom_sandbox/overlay.py) — merge, map, rewrite
- [`config/profiles/ollama.yaml`](../config/profiles/ollama.yaml)
- [`config/models.yaml`](../config/models.yaml)
- [`config/taxonomy.overlay.yaml`](../config/taxonomy.overlay.yaml)
- [`config/components.yaml`](../config/components.yaml)

**Offline / LLM-free.** Mock machinery only. Live Ollama is
`sandbox cutover --local` after `sandbox up` — not this notebook.


In [ ]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    """Walk up from cwd (hostile kernels start in notebooks/)."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "reports").is_dir():
            return cand
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

from _lib import bootstrap, isolate_outputs

ROOT = bootstrap(ROOT)
OUT = isolate_outputs(ROOT)
print("repo root :", ROOT)
print("sys.path[0]:", sys.path[0])
print("notebook outputs ->", OUT)


## Why `DEFAULT_PROVIDER=ollama` is not enough

Mailroom's shipped taxonomy still names OpenRouter champion ids such as
`qwen/qwen3.7-flash`. Pointing the base URL at Ollama without rewriting the
`model` field makes Ollama try to pull a slash-id it does not have.
[`overlay.map_model`](../src/mailroom_sandbox/overlay.py) is the map.


In [ ]:
from mailroom_sandbox.overlay import (
    list_profiles,
    load_profile,
    map_model,
    serving_family,
    build_merged_taxonomy,
    agent_assignments,
    parse_agent_models,
)
from mailroom_sandbox.providers import endpoints_for

print("profiles:", list_profiles())
ollama = load_profile("ollama")
openrouter = load_profile("openrouter")
print("ollama provider / default_model:", ollama.get("provider"), ollama.get("default_model"))
print("openrouter is opt-in:", openrouter.get("provider") == "openrouter")
print("endpoints ollama:", endpoints_for(ollama).base_url)
print()
print("champion qwen/qwen3.7-flash -> ollama tag:", map_model("qwen/qwen3.7-flash", "ollama"))
print("same champion -> vllm tag   :", map_model("qwen/qwen3.7-flash", "vllm"))
print("serving family(ollama)      :", serving_family(ollama))


## `activate()` writes the runtime taxonomy

`activate("ollama")` deep-merges the vendored/base taxonomy + overlay +
component routing, rewrites agents, then writes
`data/runtime/taxonomy.yaml`. Isolated evals call this for you; notebooks
and tests call it explicitly **before** importing mailroom graph code.


In [ ]:
from mailroom_sandbox.runtime import activate, active

act = activate("ollama", load_env_file=False)
print("profile        :", act.profile_name)
print("taxonomy path  :", act.taxonomy_path)
print("patched config :", act.patched_config, "(False until vendored mailroom is on sys.path)")
print("n agents       :", len(act.assignments))
print()
print(f"{'agent':32s} {'provider':12s} model")
for name, provider, model in act.assignments[:12]:
    print(f"{name:32s} {provider:12s} {model}")
print("...")
judge = next(row for row in act.assignments if row[0] == "judge")
sorter = next(row for row in act.assignments if row[0] == "sorter")
print("sorter ->", sorter)
print("judge  ->", judge)
assert sorter[1] == "ollama"
assert "/" not in sorter[2], "local tags must not keep the OpenRouter slash-id"
print("active() is the same object:", active() is act)


## Surgical `--agent-model` wins last

`--model qwen3:8b` rewrites **every** agent. `--agent-model judge=qwen3:14b`
is the cutover knob that only touches one name. Same merge order as
[`overlay.build_merged_taxonomy`](../src/mailroom_sandbox/overlay.py):
profile rewrite → overlay knobs → surgical models.


In [ ]:
surgical = parse_agent_models(["judge=qwen3:14b"])
tax = build_merged_taxonomy(ollama, agent_models=surgical)
by_name = {n: m for n, _p, m in agent_assignments(tax)}
print("judge after surgical override:", by_name["judge"])
print("sorter unchanged            :", by_name["sorter"])
assert by_name["judge"] == "qwen3:14b"
assert by_name["sorter"] != "qwen3:14b"

# Overlay knobs (temperature / max_tokens) survive the rewrite.
from mailroom_sandbox.overlay import load_yaml
from mailroom_sandbox.paths import config_dir
overlay = load_yaml(config_dir() / "taxonomy.overlay.yaml")
print("overlay sorter.max_tokens:", (overlay.get("agents") or {}).get("sorter", {}).get("max_tokens"))
print("runtime sorter.max_tokens:", (tax.get("agents") or {}).get("sorter", {}).get("max_tokens"))


## Component gates (not a graph fork)

[`config/components.yaml`](../config/components.yaml) enables/disables
isolated evals and overlays confidence routing. It does **not** add or
remove LangGraph nodes.


In [ ]:
from mailroom_sandbox.components import load_components, is_enabled, routing_overlay

cfg = load_components()
print("retired_agents:", cfg.get("retired_agents"))
print("sorter enabled:", is_enabled("agents", "sorter"))
print("court_opinions_specialist enabled:", is_enabled("agents", "court_opinions_specialist"))
print("routing overlay fragment:")
import json
print(json.dumps(routing_overlay(cfg), indent=2))
print()
print("HONEST GAP: live 13-node topology still lives in llm-mailroom.")
print("This sandbox overlays config; it does not reimplement graph/routing.py.")
